# Convolutional Neural Networks (CNN) — Image Classification

A comprehensive guide to CNNs covering:
- **Convolution, Pooling, Padding** concepts
- **Building a CNN** from scratch with `Conv2D`, `MaxPooling2D`
- **CIFAR-10** multi-class image classification
- **Data Augmentation** with `ImageDataGenerator`
- **Filters/Feature Maps visualization**
- **Comparison**: Dense-only vs CNN architecture

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
print(f'TensorFlow version: {tf.__version__}')

## 1. Understanding Convolution — Visual Intuition

A **convolution** slides a small filter (kernel) over the image and computes element-wise multiplication + sum at each position.

Key CNN Components:
- **Conv2D**: Learns spatial filters to detect edges, textures, patterns
- **MaxPooling2D**: Downsamples feature maps, retains dominant features
- **Flatten → Dense**: Converts 2D features to 1D for classification
- **Dropout**: Prevents overfitting

In [ ]:
# Demonstrate convolution operation manually
from scipy.signal import convolve2d

# Create a simple 8x8 image
image = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 0],
    [0, 1, 0, 0, 0, 0, 1, 0],
    [0, 1, 0, 1, 1, 0, 1, 0],
    [0, 1, 0, 1, 1, 0, 1, 0],
    [0, 1, 0, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0]
], dtype=np.float32)

# Define common filters
filters = {
    'Horizontal Edge': np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]]),
    'Vertical Edge': np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]]),
    'Sharpen': np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
    'Blur': np.ones((3, 3)) / 9
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original Image')
for i, (name, kernel) in enumerate(filters.items()):
    result = convolve2d(image, kernel, mode='same')
    axes[i+1].imshow(result, cmap='gray')
    axes[i+1].set_title(name)
for ax in axes:
    ax.axis('off')
plt.suptitle('Convolution with Different Kernels', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Load & Explore CIFAR-10 Dataset

In [ ]:
# Load CIFAR-10: 60,000 32x32 color images in 10 classes
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f'Training set: {X_train.shape} | Labels: {y_train.shape}')
print(f'Test set:     {X_test.shape} | Labels: {y_test.shape}')
print(f'Pixel range:  [{X_train.min()}, {X_train.max()}]')
print(f'Classes: {class_names}')

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    idx = np.where(y_train.flatten() == i)[0][0]
    ax.imshow(X_train[idx])
    ax.set_title(class_names[i], fontsize=12)
    ax.axis('off')
plt.suptitle('CIFAR-10 Sample Images (One per Class)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Normalize pixel values to [0, 1]
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

# One-hot encode labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print(f'Normalized pixel range: [{X_train_norm.min()}, {X_train_norm.max()}]')
print(f'One-hot label shape: {y_train_cat.shape}')
print(f'Example label (class 3): {y_train_cat[np.where(y_train.flatten()==3)[0][0]]}')

## 3. Baseline — Dense-Only Network (No CNN)

First, let's build a **fully-connected (Dense-only)** network to establish a baseline. This is what MNIST notebook used — let's see why it's inadequate for images.

In [ ]:
# Baseline: Dense-only model
baseline_model = models.Sequential([
    layers.Flatten(input_shape=(32, 32, 3)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

baseline_model.compile(optimizer='adam',
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

print(f'Baseline params: {baseline_model.count_params():,}')
baseline_model.summary()

In [ ]:
baseline_history = baseline_model.fit(
    X_train_norm, y_train_cat,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

baseline_loss, baseline_acc = baseline_model.evaluate(X_test_norm, y_test_cat, verbose=0)
print(f'\nBaseline Test Accuracy: {baseline_acc:.4f}')

## 4. CNN Architecture — Conv2D + MaxPooling2D

Now let's build a proper CNN:

```
Input (32x32x3)
  → Conv2D(32, 3x3) → ReLU → Conv2D(32, 3x3) → ReLU → MaxPool(2x2) → Dropout(0.25)
  → Conv2D(64, 3x3) → ReLU → Conv2D(64, 3x3) → ReLU → MaxPool(2x2) → Dropout(0.25)
  → Flatten → Dense(512) → ReLU → Dropout(0.5)
  → Dense(10, softmax)
```

In [ ]:
cnn_model = models.Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Classifier
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'CNN params: {cnn_model.count_params():,}')
cnn_model.summary()

In [ ]:
# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

cnn_history = cnn_model.fit(
    X_train_norm, y_train_cat,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

cnn_loss, cnn_acc = cnn_model.evaluate(X_test_norm, y_test_cat, verbose=0)
print(f'\nCNN Test Accuracy: {cnn_acc:.4f}')

## 5. Data Augmentation — Improving Generalization

In [ ]:
# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)
datagen.fit(X_train_norm)

# Visualize augmented images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
sample_img = X_train_norm[0:1]
axes[0][0].imshow(X_train_norm[0])
axes[0][0].set_title('Original')
for i, ax in enumerate(axes.flat[1:]):
    augmented = datagen.flow(sample_img, batch_size=1)
    ax.imshow(next(augmented)[0])
    ax.set_title(f'Augmented {i+1}')
for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Data Augmentation Examples', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Train CNN with augmentation
cnn_aug_model = keras.models.clone_model(cnn_model)
cnn_aug_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

aug_history = cnn_aug_model.fit(
    datagen.flow(X_train_norm, y_train_cat, batch_size=64),
    epochs=50,
    validation_data=(X_test_norm, y_test_cat),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

aug_loss, aug_acc = cnn_aug_model.evaluate(X_test_norm, y_test_cat, verbose=0)
print(f'\nCNN + Augmentation Test Accuracy: {aug_acc:.4f}')

## 6. Comparison — Dense vs CNN vs CNN+Augmentation

In [ ]:
# Training curves comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for hist, label in [(baseline_history, 'Dense-Only'), (cnn_history, 'CNN'), (aug_history, 'CNN+Aug')]:
    axes[0].plot(hist.history['accuracy'], label=f'{label} Train')
    axes[0].plot(hist.history['val_accuracy'], '--', label=f'{label} Val')

axes[0].set_title('Accuracy Comparison')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Final accuracy bar chart
models_names = ['Dense-Only', 'CNN', 'CNN + Aug']
accs = [baseline_acc, cnn_acc, aug_acc]
colors = ['#e74c3c', '#3498db', '#2ecc71']
bars = axes[1].bar(models_names, accs, color=colors, edgecolor='black')
for bar, acc in zip(bars, accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{acc:.3f}', ha='center', fontsize=12, fontweight='bold')
axes[1].set_title('Test Accuracy Comparison')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Feature Map Visualization — What the CNN "Sees"

In [ ]:
# Visualize feature maps from first conv layer
layer_outputs = [layer.output for layer in cnn_model.layers if 'conv2d' in layer.name]
feature_model = models.Model(inputs=cnn_model.input, outputs=layer_outputs[:3])

test_img = X_test_norm[0:1]  # Single test image
feature_maps = feature_model.predict(test_img, verbose=0)

fig, axes = plt.subplots(3, 8, figsize=(20, 8))
for layer_idx, fmap in enumerate(feature_maps):
    for i in range(8):
        axes[layer_idx][i].imshow(fmap[0, :, :, i], cmap='viridis')
        axes[layer_idx][i].axis('off')
    axes[layer_idx][0].set_ylabel(f'Conv Layer {layer_idx+1}', fontsize=12)
plt.suptitle('Feature Maps Across CNN Layers (First 8 filters)', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Confusion Matrix & Classification Report

In [ ]:
# Predictions and evaluation
best_model = cnn_aug_model
y_pred = np.argmax(best_model.predict(X_test_norm, verbose=0), axis=1)
y_true = y_test.flatten()

print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — CNN + Augmentation')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## 9. Key Takeaways

| Concept | Details |
|---------|--------|
| **Conv2D** | Learns spatial hierarchies — edges → textures → objects |
| **MaxPooling2D** | Reduces spatial dimensions, adds translation invariance |
| **BatchNormalization** | Stabilizes & accelerates training |
| **Data Augmentation** | Artificially expands dataset, reduces overfitting |
| **Dense vs CNN** | CNN captures spatial structure; Dense networks cannot |
| **Feature Maps** | Early layers detect edges; deeper layers detect complex patterns |

### Architecture Design Tips
- Start with small filters (3×3), increase filter count deeper in the network
- Use `padding='same'` to preserve spatial dimensions
- Apply BatchNorm after Conv2D, before activation (or after — both work)
- Use Dropout for regularization (lower rates in conv blocks, higher in dense)
- Always use Data Augmentation for small datasets